# 03 - Segmentation avec U-Net 3D

Ce notebook vérifie le modèle U-Net 3D, calcule les métriques et fournit une base d'inférence. L'entraînement complet est volontairement séparé afin de ne pas lancer automatiquement une expérience coûteuse.

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from brain_mri_fem.segmentation import UNet3D, dice, iou
from brain_mri_fem.utils import seed_everything

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet3D(in_channels=1, out_channels=2, channels=(8, 16, 32)).to(device)
print("Device:", device)
print("Paramètres:", sum(parameter.numel() for parameter in model.parameters()))

Device: cpu
Paramètres: 85026


In [ ]:
volume = torch.randn(1, 1, 32, 32, 32, device=device)
with torch.no_grad():
    logits = model(volume)
prediction = logits.argmax(dim=1).cpu().numpy()[0]
print("Entrée:", tuple(volume.shape), "Sortie:", tuple(logits.shape))
print("Classes prédites:", np.unique(prediction).tolist())

target = np.zeros_like(prediction)
target[10:22, 10:22, 10:22] = 1
print("Dice exemple:", dice(prediction, target))
print("IoU exemple:", iou(prediction, target))

In [ ]:
slice_index = prediction.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(volume[0, 0, :, :, slice_index].cpu(), cmap="gray")
axes[0].set_title("Volume d'entrée")
axes[1].imshow(prediction[:, :, slice_index], cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Prédiction argmax")
for axis in axes:
    axis.axis("off")
fig.tight_layout()
output = REPO_ROOT / "results/figures/segmentation"
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "unet3d_smoke_test.png", dpi=150)
plt.show()